# Portfolio accounting and independent rebalance
Saved retrospective research. No training or held-out reselection. Run from the repository root or notebooks directory.


In [1]:
from pathlib import Path
import json,sys
r=Path.cwd()
if not (r/'artifacts/research').exists(): r=r.parent
sys.path.insert(0,str(r/'src'))
b=json.loads((r/'artifacts/research/public_bundle.json').read_text())
a=b['analysis']


## First cash rebalance; independently implemented

In [2]:
v=b["review"];cash=v["initial_nav"];mv=0
for x in v["assets"]:
 price=x["raw_open"]*(1+v["slippage_bps"]/10000)
 q=min(int(v["initial_nav"]*x["target_weight"]/x["raw_open"]),int(x["volume"]*v["participation"]),int(max(cash,0)/(price*(1+v["fee_bps"]/10000))))
 cash-=q*price*(1+v["fee_bps"]/10000);mv+=q*x["raw_close"]
 print(x["asset"],q)
assert abs(cash+mv-v["python_ledger"]["nav"])<1e-7
print("NAV",cash+mv,"cash",cash)

XLB 0
XLE 383
XLF 0
XLI 0
XLK 265
XLP 0
XLU 0
XLV 0
XLY 256
NAV 97918.169238186 cash 63.789238186007424


## Asset P&L reconciliation

In [3]:
for n,x in a["strategies"].items():
 assert abs(x["attribution_total_difference"])<1e-6
 print(n,x["cumulative_net_pnl_by_asset"],x["attribution_max_daily_residual"])

equal_weight {'XLB': 2887.0735771750033, 'XLE': 2286.4196482650023, 'XLF': 7327.202445383501, 'XLI': 6817.598130180001, 'XLK': 11509.229918849998, 'XLP': 1681.5384434475002, 'XLU': 4403.436164277002, 'XLV': 2838.2344986225, 'XLY': 8704.399702506003} 4.547473508864641e-11
momentum_63 {'XLB': 1396.6954991670036, 'XLE': -8894.503252336992, 'XLF': 8163.43589237151, 'XLI': 3740.098497734527, 'XLK': 25976.447525202013, 'XLP': -622.7135311539837, 'XLU': 6877.6253785460485, 'XLV': -2564.601675205499, 'XLY': 10678.700590639019} 2.9103830456733704e-11
inverse_vol {'XLB': 2084.815238273003, 'XLE': 670.5341337660055, 'XLF': 7129.164659047006, 'XLI': 7098.735837288508, 'XLK': 9571.689989814004, 'XLP': 1872.4547977750033, 'XLU': 3665.7266638130027, 'XLV': 2991.591463758505, 'XLY': 6557.518618351002} 2.7284841053187847e-11
stumps_24 {'XLB': 2928.8028223990173, 'XLE': 7470.741764590031, 'XLF': 1792.9769970345103, 'XLI': 2979.163501575017, 'XLK': 10842.238523145039, 'XLP': -1901.0396574474853, 'XLU': 3